# 3.3 同一個位置，為什麼要有查詢與匹配兩份表示？


一個位置既會問「我現在需要哪些線索」，又會成為別的位置可讀的資料。這兩份工作不必用同一套數字：**query 表示提出需求，key 表示提供匹配線索**，所以模型為它們準備不同的可調配方。

例如已有特徵 `[1,2]`，手動指定 query 配方保留兩格，key 配方交換兩格，就分別得到 `[1,2]` 與 `[2,1]`。它們來自同一份資訊，卻可因不同配方扮演不同角色。這是數字轉換，並不是先為位置貼上某個文法標籤。

下面先直接給轉換後的卡：兩個查詢與三個候選。所有查詢排成表 Q，所有 key 排成表 K。希望每個查詢各自比較全部三候選，得到兩列三欄的分數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
scores = q @ k.T
print("Q", q.shape, "K轉置", k.T.shape)
print(scores)
assert scores.shape == (2, 3)

第一列 query `[1,0]` 得 `[1,0,1]`，第二列 `[0,1]` 得 `[0,1,1]`。第三個 key `[1,1]` 對兩種需求都得 1。輸出每列是一個問的人，每欄是一個可讀位置；`k.T` 只改排列以便相乘，不會把 key 改成 query。

用尺寸核對：Q 是 `[2,2]`，K 轉置是 `[2,3]`，共同的內側 2 是匹配特徵數，輸出 `[2,3]`。查詢數與候選數不必相等，但兩邊的匹配特徵數要相同。

真實模型讓位置表示 X 各經過一套 Linear，得到 Q 與 K；這種特徵轉換也叫**投影**，這裡先理解為不同的加權配方。**自注意力**的「自」是指：提出查詢、提供匹配線索，以及實際取回的內容，都由同一組位置產生。這裡先看 Q、K 的匹配部分；下一節會加入代表取回內容的第三份表示。位置來源相同，角色仍不同。

練習在 K 最後加 `[2,0]`，輸出變兩列四欄，新欄為 2、0。改斷言為 `(2,4)` 再核對。若再做 softmax，要沿每列的候選欄，讓每個問題自己分配讀取比例。

<details>
<summary>補充與重做</summary>

Q `[Tq,Dk]`、K `[Tk,Dk]` 得分數 `[Tq,Tk]`，Tq 是查詢數，Tk 是候選數，Dk 是匹配特徵數。不同查詢之間不一起做 softmax，否則會讓不同問題互搶比例。Linear 見 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)；本例直接指定 Q/K，沒有建立或訓練投影權重。

</details>
